# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides an example for loading and exploring a dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described via a Croissant schema at this URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure that mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {getattr(metadata, 'name', '')}\nDescription: {getattr(metadata, 'description', '')}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

--

**Note:** The Croissant schema for this dataset contains tabular data, where record sets represent top-level tables. We will list all record set `@id`s and, for each, list their field and column `@id`s and names.

In [ ]:
# List all record sets and their fields using @id references
record_sets = dataset.record_sets()

print(f"There are {len(record_sets)} record sets in this dataset.\n")
for rs in record_sets:
    print(f"RecordSet name: {getattr(rs, 'name', '')}")
    print(f"  @id: {rs.id}")
    # List fields (columns) within each record set
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {getattr(field, 'name', '')} (@id: {field.id})")
    print()

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s as shown above. We'll load **all** record sets into separate DataFrames, using their `@id` as keys.

In [ ]:
# Extract the data for each record set
dataframes = {}

for rs in record_sets:
    rs_id = rs.id
    records_iter = dataset.records(record_set=rs_id)
    records = list(records_iter)
    if records:  # Only store if not empty
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"RecordSet '{rs_id}' loaded: {dataframes[rs_id].shape[0]} records, {dataframes[rs_id].shape[1]} columns.")
    else:
        print(f"RecordSet '{rs_id}' is empty or could not be loaded.")

# Show example columns and first few rows for the first non-empty record set
main_record_set_id = None
for rs_id, df in dataframes.items():
    main_record_set_id = rs_id
    print(f"\nColumns for record set '@id: {rs_id}':")
    print(df.columns.tolist())
    print("\nExample rows:")
    display(df.head())  # Only display for the first one
    break

# Store for later use
# 'main_record_set_id' is the @id of the record set used for demo


## 4. Exploratory Data Analysis (EDA)

We will:

- Select a numeric field for analysis (check for integer or float columns).
- Filter for records where that field exceeds a threshold.
- Normalize the numeric field.
- Optionally, group results by a categorical field if available (e.g., MSI-H status or anatomical site).

In [ ]:
# Pick a numeric field in the main record set for filtering and normalization
df = dataframes[main_record_set_id]

numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

if numeric_field:
    print(f"Using numeric field for EDA: '{numeric_field}'")

    # Choose an appropriate threshold (e.g., mean or arbitrary)
    threshold = df[numeric_field].mean() if pd.api.types.is_float_dtype(df[numeric_field]) else 10

    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"\nNormalized '{numeric_field}' values:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to find a categorical or grouping field (e.g., 'MSI_status', 'sex', etc.)
    group_field = None
    # Heuristic - select the first non-numeric column
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            group_field = col
            break
    if group_field:
        print(f"\nGrouping by categorical field: '{group_field}'")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        display(grouped_df.head())
else:
    print("No numeric field available for EDA in this record set.")

## 5. Visualization

Visualize numeric distributions and categorical breakdowns.

- Plot a histogram of the selected numeric field.
- Show a boxplot of the numeric variable grouped by a categorical field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"'{numeric_field}' by '{group_field}'")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric data to visualize in this record set.")

## 6. Conclusion

We successfully loaded and explored the data from the FAIR$^2$ Clinicopathological CRC Survivors dataset using the Croissant schema and the `mlcroissant` library.

- We listed all record sets and their field `@id`s.
- Loaded tabular data into pandas DataFrames for analysis.
- Performed simple exploratory data analysis, filtering, normalization, grouping, and visualization.

**Next steps:**
- More in-depth modeling or statistical hypothesis testing on the clinical variables
- Cross-record set linking (if multiple related tables present)
- Export processed results or train predictive models

*For further details on the FAIR$^2$ dataset and its clinical context, see the [dataset description on sen.science](https://sen.science/doi/10.71728/senscience.qs2f-h81p).*